# Notebook 01 — Business Understanding

**Project:** Credit Risk Analytics   
**Date:** July 2026  
**Dataset:** Home Credit Default Risk (Kaggle)  

---

## 1.1 Business Problem

The lending platform offers unsecured personal loans ($1,000-$35,000, 12-60 month terms) to near-prime borrowers (credit scores 580-680). This segment is underserved by traditional banks but carries elevated default risk.

**The core problem:** Each percentage-point improvement in default prediction accuracy translates to millions in avoided losses. Horizon needs a data-driven model to:
- Identify high-risk applicants before funding.
- Reduce net charge-offs by 15–20%.
- Price loans appropriately for each risk segment.
- Prioritise collections on accounts most likely to default.

**Business impact:** A 5-point AUC improvement (0.75 → 0.80) is estimated to save $2–3M annually.

---

## 1.2 Dataset Overview

We use the [Home Credit Default Risk](https://www.kaggle.com/competitions/home-credit-default-risk/data) dataset from Kaggle. It contains ~300,000 loan applicants with ~120 features across 8 interconnected tables. The dataset mimics a real-world lending environment where:
- Applicant demographics and loan details are in the main table.
- External credit bureau scores are available for most applicants.
- Past loan repayment history is recorded across multiple auxiliary tables.
- The default rate is ~8%, reflecting a realistic near-prime portfolio.

### Key Tables

| Table | Rows (est.) | Description |
|---|---|---|
| `application_train` | ~307K | Main applicant data with target label |
| `application_test` | ~48K | Test set (no target) |
| `bureau` | ~1.7M | Credit bureau data per applicant |
| `bureau_balance` | ~14M | Monthly bureau balance snapshots |
| `previous_application` | ~1.6M | Previous loan applications at Home Credit |
| `installments_payments` | ~13.6M | Payment history on previous loans |
| `POS_CASH_balance` | ~10M | POS/cash loan monthly balances |
| `credit_card_balance` | ~3.8M | Credit card monthly balances |

---

## 1.3 Prediction Target

**Target variable:** `TARGET`  
- `1` = Applicant defaulted on the loan.
- `0` = Applicant repaid the loan.

**Definition per Home Credit:** A borrower is marked as defaulted if they had a payment delay of more than X days on their first loan installment. This aligns with the industry-standard "90+ days past due" definition.

**Default rate:** ~8% (imbalanced, as expected for a near-prime portfolio).

---

## 1.4 Business Objectives

1. **Primary:** Build a default-probability model that achieves AUC ≥ 0.80 on a holdout test set.
2. **Secondary:** Translate model outputs into dollar-denominated profit simulations.
3. **Tertiary:** Produce an interactive dashboard for stakeholders.
4. **Compliance:** Perform fairness analysis to ensure no disparate impact.

---

## 1.5 Stakeholder Map

| Stakeholder | Concern |
|---|---|
| Chief Risk Officer (CRO) | "Is the model sound, fair, and compliant?" |
| Underwriting Team | "Can we trust this score to automate decisions?" |
| Collections Manager | "Which accounts should we call first?" |
| Finance / Treasury | "What is our expected loss next quarter?" |
| Regulatory Compliance | "Does the model discriminate against protected groups?" |
| Product Manager | "Can we offer better rates to good borrowers?" |

---

## 2. Setup and Imports

In [1]:
# ============================================================================
# 2. SETUP AND IMPORTS
# ============================================================================
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import numpy as np
import pandas as pd

# Paths
# Find project root (works from notebooks/ or notebooks/execution/)
_cwd = os.path.abspath(os.getcwd())
for _ in range(4):
    if os.path.exists(os.path.join(_cwd, 'requirements.txt')):
        break
    _cwd = os.path.dirname(_cwd)
ROOT = _cwd
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')

# Reproducibility
np.random.seed(42)

print('Imports loaded successfully.')
print(f'Project root: {PROJECT_ROOT}')

Imports loaded successfully.
Project root: /Users/haoquanzhang/Desktop/credit-risk-analytics


## 3. Load the Dataset

The Home Credit dataset must be downloaded from Kaggle. The main file is `application_train.csv`. We check for its existence and load it.

In [2]:
# ============================================================================
# 3. LOAD THE DATASET
# ============================================================================
# Expected file path
train_path = os.path.join(DATA_RAW_DIR, 'application_train.csv')

if not os.path.exists(train_path):
    print('=' * 70)
    print('  DATASET NOT FOUND IN data/raw/')
    print('=' * 70)
    print('\nThe Home Credit Default Risk dataset is required.')
    print('\nDownload steps:')
    print('  1. Visit: https://www.kaggle.com/competitions/home-credit-default-risk/data')
    print('  2. Download application_train.csv, application_test.csv, and auxiliary tables.')
    print(f'  3. Place all CSV files in: {DATA_RAW_DIR}')
    print('\nFor automated download, install kagglehub:')
    print('  pip install kagglehub')
    print('  import kagglehub')
    print('  path = kagglehub.dataset_download("competitions/home-credit-default-risk")')
    print('=' * 70)
    raise FileNotFoundError(f'Dataset not found at {train_path}')

df = pd.read_csv(train_path)
print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

Dataset loaded: 307,511 rows × 122 columns


## 4. Data Structure Description

In [3]:
# ============================================================================
# 4. DATA STRUCTURE OVERVIEW
# ============================================================================
print('\n=== First 5 rows ===')
display(df.head(3))

print('\n=== Data types ===')
dtype_counts = df.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f'  {dtype}: {count} columns')

print(f'\n=== Basic info ===')
print(f'  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'  Numeric columns: {df.select_dtypes(include=[np.number]).shape[1]}')
print(f'  Categorical columns: {df.select_dtypes(include=["object", "category"]).shape[1]}')


=== First 5 rows ===


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0



=== Data types ===
  float64: 65 columns
  int64: 41 columns
  str: 16 columns

=== Basic info ===


  Memory usage: 504.99 MB


  Numeric columns: 106
  Categorical columns: 16


In [4]:
# ============================================================================
# 4.1 COLUMN GROUPS BY THEME
# ============================================================================
# Group columns by their prefix/suffix to understand the schema
column_groups = {
    'Applicant Demographics': [c for c in df.columns if c.startswith(('NAME_', 'CODE_','CNT_'))],
    'Financial': [c for c in df.columns if c.startswith('AMT_')],
    'External Risk Scores': [c for c in df.columns if c.startswith('EXT_SOURCE')],
    'Days / Temporal': [c for c in df.columns if c.startswith('DAYS_')],
    'Flags & Indicators': [c for c in df.columns if c.startswith(('FLAG_', 'REG_'))],
    'Bureau Requests': [c for c in df.columns if 'AMT_REQ' in c],
    'Target': ['TARGET'],
}

print('=== Column Groups ===')
for group_name, cols in column_groups.items():
    print(f'\n{group_name} ({len(cols)} columns):')
    for c in cols[:6]:
        print(f'    {c}')
    if len(cols) > 6:
        print(f'    ... and {len(cols) - 6} more')

=== Column Groups ===

Applicant Demographics (9 columns):
    NAME_CONTRACT_TYPE
    CODE_GENDER
    CNT_CHILDREN
    NAME_TYPE_SUITE
    NAME_INCOME_TYPE
    NAME_EDUCATION_TYPE
    ... and 3 more

Financial (10 columns):
    AMT_INCOME_TOTAL
    AMT_CREDIT
    AMT_ANNUITY
    AMT_GOODS_PRICE
    AMT_REQ_CREDIT_BUREAU_HOUR
    AMT_REQ_CREDIT_BUREAU_DAY
    ... and 4 more

External Risk Scores (3 columns):
    EXT_SOURCE_1
    EXT_SOURCE_2
    EXT_SOURCE_3

Days / Temporal (5 columns):
    DAYS_BIRTH
    DAYS_EMPLOYED
    DAYS_REGISTRATION
    DAYS_ID_PUBLISH
    DAYS_LAST_PHONE_CHANGE

Flags & Indicators (32 columns):
    FLAG_OWN_CAR
    FLAG_OWN_REALTY
    FLAG_MOBIL
    FLAG_EMP_PHONE
    FLAG_WORK_PHONE
    FLAG_CONT_MOBILE
    ... and 26 more

Bureau Requests (6 columns):
    AMT_REQ_CREDIT_BUREAU_HOUR
    AMT_REQ_CREDIT_BUREAU_DAY
    AMT_REQ_CREDIT_BUREAU_WEEK
    AMT_REQ_CREDIT_BUREAU_MON
    AMT_REQ_CREDIT_BUREAU_QRT
    AMT_REQ_CREDIT_BUREAU_YEAR

Target (1 columns):
    TA

## 5. Target Variable Analysis

In [5]:
# ============================================================================
# 5. TARGET VARIABLE ANALYSIS
# ============================================================================
target_counts = df['TARGET'].value_counts(normalize=True)
target_abs = df['TARGET'].value_counts()

print('=== Target Distribution ===')
print(f'\n  0 — Repaid:   {target_abs[0]:>8,}  ({target_counts[0]:.2%})')
print(f'  1 — Defaulted: {target_abs[1]:>8,}  ({target_counts[1]:.2%})')
print(f'\n  Total:         {len(df):>8,}')
print(f'  Default rate:  {target_counts[1]:.2%}')

print('\n  → The 8% default rate is consistent with the near-prime segment.')
print('  → Class imbalance is expected and natural.')
print('  → Accuracy alone will be misleading; use AUC, precision, and recall.')

=== Target Distribution ===

  0 — Repaid:    282,686  (91.93%)
  1 — Defaulted:   24,825  (8.07%)

  Total:          307,511
  Default rate:  8.07%

  → The 8% default rate is consistent with the near-prime segment.
  → Class imbalance is expected and natural.
  → Accuracy alone will be misleading; use AUC, precision, and recall.


## 6. Key Feature Groups — Business Interpretation

### Applicant Profile
Features capturing who the borrower is — age, income, employment, family situation. These measure **capacity to repay**.

### External Risk Scores (`EXT_SOURCE_1`, `2`, `3`)
Normalised credit scores from third-party bureaus. These are typically the **single most predictive feature group** — they summarise the applicant's creditworthiness as assessed by external agencies.

### Financial Amounts (`AMT_*`)
Loan amount, annuity (monthly payment), income, and requested credit. These determine **debt burden**.

### Temporal Features (`DAYS_*`)
Age, employment length, and registration recency. Capture **stability** — longer employment and older age generally mean lower risk.

### Flags (`FLAG_*`, `REG_*`)
Binary indicators for phone, email, and address verification. These measure **data completeness and identity verification**.

### Bureau Data (separate tables)
Historical credit activity outside Home Credit — number of past credits, outstanding debt, delinquencies. Captures **external indebtedness**.

In [6]:
# ============================================================================
# 6.1 SELECTED FEATURE PREVIEW
# ============================================================================
key_features = [
    'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'AMT_INCOME_TOTAL',
    'AMT_CREDIT', 'AMT_ANNUITY', 'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS', 'DAYS_BIRTH', 'DAYS_EMPLOYED',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

available_keys = [c for c in key_features if c in df.columns]
print('=== Key Features Preview (first 5 rows) ===')
display(df[available_keys].head(5))

=== Key Features Preview (first 5 rows) ===


,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,DAYS_BIRTH,DAYS_EMPLOYED,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
0,1,Cash loans,M,202500.0,406597.5,24700.5,Secondary / secondary special,Single / not married,-9461,-637,0.083037,0.262949,0.139376
1,0,Cash loans,F,270000.0,1293502.5,35698.5,Higher education,Married,-16765,-1188,0.311267,0.622246,NaN
2,0,Revolving loans,M,67500.0,135000.0,6750.0,Secondary / secondary special,Single / not married,-19046,-225,NaN,0.555912,0.729567
3,0,Cash loans,F,135000.0,312682.5,29686.5,Secondary / secondary special,Civil marriage,-19005,-3039,NaN,0.650442,NaN
4,0,Cash loans,M,121500.0,513000.0,21865.5,Secondary / secondary special,Single / not married,-19932,-3038,NaN,0.322738,NaN


## 7. Business Objectives Recap

This notebook established:

1. **The business problem:** Predict loan default to reduce losses and improve portfolio profitability.
2. **The dataset:** Home Credit Default Risk with ~307K applicants and 8 interconnected tables.
3. **The target:** `TARGET` (1 = default, 0 = repaid) with an 8% default rate.
4. **The stakeholders:** Risk, Underwriting, Collections, Finance, Compliance, Product.
5. **Success criteria:** AUC ≥ 0.80, profit simulation, dashboard, fairness analysis.

**Next notebook:** Exploratory Data Analysis — visualising patterns, checking data quality, and generating hypotheses.